# AquaEye: Exploratory Data Analysis (EDA)
### Environmental Metadata & Dynamic Parameter Correlation Analysis

This notebook explores the five metadata streams captured alongside aquatic shrimp videos:
1. **Depth Estimation** (`depth_estimation.csv`)
2. **Light Reflection & Backscatter** (`light_reflection.csv`)
3. **Multiscale Image Tiling** (`multiscale_image_tiling.csv`)
4. **Sensory Enhancement Parameters** (`sensory_enhancement.csv`)
5. **Spatial Awareness & Clustering** (`spatial_awareness.csv`)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="darkgrid", palette="mako")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 10

# Resolve metadata path
METADATA_DIR = Path("../data/metadata").resolve()
print(f"Loading metadata from: {METADATA_DIR}")

## 1. Load Individual CSV Datasets

In [ ]:
df_depth = pd.read_csv(METADATA_DIR / "depth_estimation.csv")
df_light = pd.read_csv(METADATA_DIR / "light_reflection.csv")
df_tiling = pd.read_csv(METADATA_DIR / "multiscale_image_tiling.csv")
df_sensory = pd.read_csv(METADATA_DIR / "sensory_enhancement.csv")
df_spatial = pd.read_csv(METADATA_DIR / "spatial_awareness.csv")

print(f"Depth Estimation records: {len(df_depth)}")
print(f"Light Reflection records: {len(df_light)}")
print(f"Multiscale Tiling records: {len(df_tiling)}")
print(f"Sensory Enhancement records: {len(df_sensory)}")
print(f"Spatial Awareness records: {len(df_spatial)}")

## 2. Preview Depth & Water Turbidity Data

In [ ]:
df_depth.head(10)

## 3. Merge Timeseries & Sensory Metadata

In [ ]:
# Merge temporal tables on video_id and timestamp_s
df_merged = pd.merge(df_depth, df_light, on=["video_id", "timestamp_s"], how="inner")
df_merged = pd.merge(df_merged, df_sensory, on=["video_id", "timestamp_s"], how="inner")
df_merged = pd.merge(df_merged, df_spatial, on="video_id", how="left")

print(f"Master joined shape: {df_merged.shape}")
df_merged.head()

## 4. Relationship Between Turbidity and Dynamic CLAHE Enhancement

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Turbidity vs CLAHE Clip Limit
sns.scatterplot(
    data=df_merged,
    x="water_clarity_turbidity_ntu",
    y="clahe_clip_limit",
    hue="estimated_depth_m",
    size="denoise_h",
    palette="viridis",
    ax=ax[0]
)
ax[0].set_title("Turbidity (NTU) vs. Dynamic CLAHE Clip Limit", fontweight="bold")
ax[0].set_xlabel("Water Clarity / Turbidity (NTU)")
ax[0].set_ylabel("CLAHE Clip Limit")

# Depth vs Ambient Lux
sns.lineplot(
    data=df_merged,
    x="estimated_depth_m",
    y="ambient_lux",
    marker="o",
    color="teal",
    ax=ax[1]
)
ax[1].set_title("Water Depth (m) vs. Ambient Illumination (Lux)", fontweight="bold")
ax[1].set_xlabel("Estimated Depth (meters)")
ax[1].set_ylabel("Ambient Lux")

plt.tight_layout()
plt.show()

## 5. Spatial DBSCAN Parameters by Pond Zone

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(
    data=df_spatial,
    x="pond_zone",
    y="dbscan_eps",
    hue="density_category",
    dodge=False,
    ax=ax
)
plt.title("DBSCAN Epsilon Radius Across Different Pond Zones", fontweight="bold")
plt.xlabel("Pond Ecological Zone")
plt.ylabel("DBSCAN Epsilon (pixels)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap across Sensor Features

In [ ]:
numeric_cols = [
    "estimated_depth_m",
    "water_clarity_turbidity_ntu",
    "glint_intensity",
    "backscatter_coeff",
    "clahe_clip_limit",
    "denoise_h",
    "lab_l_channel_boost",
    "gamma_correction",
    "dbscan_eps",
    "dbscan_min_samples"
]

corr_matrix = df_merged[numeric_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Cross-Sensor Parameter Correlation Heatmap", fontweight="bold")
plt.tight_layout()
plt.show()